In [ ]:
# F1 Points Finish Predictor

**Goal:** Predict whether a driver will finish in the top 10 (score championship points) in a given race, using only information available before the race starts.

**Data:** 2022-2025 F1 seasons via FastF1 API, stored in PostgreSQL.

**Target:** `points_finish` (binary: 1 if finished top 10, 0 otherwise)

**Approach:** Feature engineering (qualifying performance, driver/constructor form, circuit history) + XGBoost classification, with strict time-based validation to avoid data leakage.

In [10]:
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv('../.env')

engine = create_engine(
    f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

print("Connected to database")

Connected to database


In [11]:
pd.read_sql("SELECT season, COUNT(*) as races FROM races GROUP BY season ORDER BY season", engine)

,season,races
0,2022,22
1,2023,22
2,2024,24
3,2025,24


In [12]:
query = """
SELECT 
    rr.driver_id,
    rr.constructor_id,
    rr.grid_position,
    rr.finish_position,
    rr.points,
    rr.points_finish,
    rr.status,
    r.season,
    r.round,
    r.circuit_id,
    r.date,
    q.q1_time,
    q.q2_time,
    q.q3_time,
    w.air_temp,
    w.track_temp,
    w.rainfall,
    w.humidity
FROM race_results rr
JOIN races r ON rr.race_id = r.race_id
LEFT JOIN qualifying_results q ON q.race_id = rr.race_id AND q.driver_id = rr.driver_id
LEFT JOIN weather w ON w.race_id = rr.race_id
ORDER BY r.date, rr.finish_position
"""

df = pd.read_sql(query, engine)
print(df.shape)
df.head()

(1838, 18)


,driver_id,constructor_id,grid_position,finish_position,points,points_finish,status,season,round,circuit_id,date,q1_time,q2_time,q3_time,air_temp,track_temp,rainfall,humidity
0,leclerc,ferrari,1.0,1.0,26.0,True,Finished,2022,1,Sakhir,2022-03-20,91.471,90.932,90.558,23.617791,28.610429,False,29.490798
1,sainz,ferrari,3.0,2.0,18.0,True,Finished,2022,1,Sakhir,2022-03-20,91.567,90.787,90.687,23.617791,28.610429,False,29.490798
2,hamilton,mercedes,5.0,3.0,15.0,True,Finished,2022,1,Sakhir,2022-03-20,92.285,91.048,91.238,23.617791,28.610429,False,29.490798
3,russell,mercedes,9.0,4.0,12.0,True,Finished,2022,1,Sakhir,2022-03-20,92.269,91.252,92.216,23.617791,28.610429,False,29.490798
4,kevin_magnussen,haas,7.0,5.0,10.0,True,Finished,2022,1,Sakhir,2022-03-20,91.955,91.461,91.808,23.617791,28.610429,False,29.490798


In [13]:
pd.read_sql("""
    SELECT driver_id, race_id, COUNT(*) as count 
    FROM race_results 
    GROUP BY driver_id, race_id 
    HAVING COUNT(*) > 1
    LIMIT 5
""", engine)

,driver_id,race_id,count


In [14]:
df.isnull().sum()

driver_id            0
constructor_id       0
grid_position        2
finish_position      2
points               0
points_finish        0
status               0
season               0
round                0
circuit_id           0
date                 0
q1_time             21
q2_time            476
q3_time            941
air_temp             0
track_temp           0
rainfall             0
humidity             0
dtype: int64

In [16]:
df['made_q2'] = df['q2_time'].notna().astype(int)

In [17]:
df['made_q3'] = df['q3_time'].notna().astype(int)

In [18]:
df[['q2_time', 'made_q2', 'q3_time', 'made_q3']].head(15)

,q2_time,made_q2,q3_time,made_q3
0,90.932,1,90.558,1
1,90.787,1,90.687,1
2,91.048,1,91.238,1
3,91.252,1,92.216,1
4,91.461,1,91.808,1
5,91.717,1,91.560,1
6,91.782,1,NaN,0
7,NaN,0,NaN,0
8,91.621,1,92.195,1
9,93.543,1,NaN,0


In [19]:
df.columns.tolist()

['driver_id',
 'constructor_id',
 'grid_position',
 'finish_position',
 'points',
 'points_finish',
 'status',
 'season',
 'round',
 'circuit_id',
 'date',
 'q1_time',
 'q2_time',
 'q3_time',
 'air_temp',
 'track_temp',
 'rainfall',
 'humidity',
 'made_q2',
 'made_q3']

In [20]:
pole_times = df[df['grid_position'] == 1]
pole_times = pole_times[['season', 'round', 'q3_time']]
pole_times = pole_times.rename(columns={'q3_time': 'pole_time'})

df = df.merge(pole_times, on=['season', 'round'], how='left')

df['quali_gap_to_pole'] = df['q3_time'] - df['pole_time']

In [21]:
df[['driver_id', 'grid_position', 'q3_time', 'pole_time', 'quali_gap_to_pole']].head(20)

,driver_id,grid_position,q3_time,pole_time,quali_gap_to_pole
0,leclerc,1.0,90.558,90.558,0.000
1,sainz,3.0,90.687,90.558,0.129
2,hamilton,5.0,91.238,90.558,0.680
3,russell,9.0,92.216,90.558,1.658
4,kevin_magnussen,7.0,91.808,90.558,1.250
5,bottas,6.0,91.560,90.558,1.002
6,ocon,11.0,NaN,90.558,NaN
7,tsunoda,16.0,NaN,90.558,NaN
8,alonso,8.0,92.195,90.558,1.637
9,zhou,15.0,NaN,90.558,NaN


In [22]:
df['quali_gap_to_pole'] = df.groupby(['season', 'round'])['quali_gap_to_pole'].transform(
    lambda x: x.fillna(x.max() + 1.0)
)

In [24]:
df = df.sort_values(['driver_id', 'date']).reset_index(drop=True)
df['driver_rolling_avg_finish'] = (
    df.groupby('driver_id')['finish_position']
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

In [25]:
df[['driver_id', 'date', 'finish_position', 'driver_rolling_avg_finish']].head(20)

,driver_id,date,finish_position,driver_rolling_avg_finish
0,albon,2022-03-20,13.0,NaN
1,albon,2022-03-27,14.0,13.000000
2,albon,2022-04-10,10.0,13.500000
3,albon,2022-04-24,11.0,12.333333
4,albon,2022-05-08,9.0,12.000000
5,albon,2022-05-22,18.0,11.400000
6,albon,2022-05-29,18.0,12.400000
7,albon,2022-06-12,12.0,13.200000
8,albon,2022-06-19,13.0,13.600000
9,albon,2022-07-03,20.0,14.000000


In [28]:
df[df['constructor_id'] == 'williams'][['driver_id', 'date', 'finish_position', 'constructor_rolling_avg_finish']].head(20)

,driver_id,date,finish_position,constructor_rolling_avg_finish
0,albon,2022-03-20,13.0,NaN
1,albon,2022-03-27,14.0,13.000000
2,albon,2022-04-10,10.0,13.500000
3,albon,2022-04-24,11.0,12.333333
4,albon,2022-05-08,9.0,12.000000
5,albon,2022-05-22,18.0,11.400000
6,albon,2022-05-29,18.0,12.400000
7,albon,2022-06-12,12.0,13.200000
8,albon,2022-06-19,13.0,13.600000
9,albon,2022-07-03,20.0,14.000000


In [30]:
df = df.sort_values(['constructor_id', 'date', 'driver_id']).reset_index(drop=True)

df['constructor_rolling_avg_finish'] = (
    df.groupby('constructor_id')['finish_position']
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

df[df['constructor_id'] == 'williams'][['driver_id', 'date', 'finish_position', 'constructor_rolling_avg_finish']].head(20)

,driver_id,date,finish_position,constructor_rolling_avg_finish
1655,albon,2022-03-20,13.0,NaN
1656,latifi,2022-03-20,16.0,13.000000
1657,albon,2022-03-27,14.0,14.500000
1658,latifi,2022-03-27,18.0,14.333333
1659,albon,2022-04-10,10.0,15.250000
1660,latifi,2022-04-10,16.0,14.200000
1661,albon,2022-04-24,11.0,14.800000
1662,latifi,2022-04-24,16.0,13.800000
1663,albon,2022-05-08,9.0,14.200000
1664,latifi,2022-05-08,14.0,12.400000


In [31]:
df[df['driver_id'] == 'albon'][['driver_id', 'date', 'finish_position', 'driver_rolling_avg_finish']].head(10)

,driver_id,date,finish_position,driver_rolling_avg_finish
1655,albon,2022-03-20,13.0,NaN
1657,albon,2022-03-27,14.0,13.000000
1659,albon,2022-04-10,10.0,13.500000
1661,albon,2022-04-24,11.0,12.333333
1663,albon,2022-05-08,9.0,12.000000
1665,albon,2022-05-22,18.0,11.400000
1667,albon,2022-05-29,18.0,12.400000
1669,albon,2022-06-12,12.0,13.200000
1671,albon,2022-06-19,13.0,13.600000
1673,albon,2022-07-03,20.0,14.000000


In [32]:
df = df.sort_values(['driver_id', 'date']).reset_index(drop=True)

df['driver_career_avg'] = (
    df.groupby('driver_id')['finish_position']
    .transform(lambda x: x.shift(1).expanding().mean())
)

df['driver_circuit_avg'] = (
    df.groupby(['driver_id', 'circuit_id'])['finish_position']
    .transform(lambda x: x.shift(1).expanding().mean())
)

df['track_delta'] = df['driver_career_avg'] - df['driver_circuit_avg']

In [33]:
df = df.sort_values(['constructor_id', 'date', 'driver_id']).reset_index(drop=True)

df['constructor_career_avg'] = (
    df.groupby('constructor_id')['finish_position']
    .transform(lambda x: x.shift(1).expanding().mean())
)

In [34]:
df['driver_vs_team_delta'] = df['driver_career_avg'] - df['constructor_career_avg']
df['driver_form_delta'] = df['driver_rolling_avg_finish'] - df['driver_career_avg']

In [35]:
df['has_circuit_history'] = df['driver_circuit_avg'].notna().astype(int)

In [36]:
feature_check = [
    'grid_position', 'quali_gap_to_pole', 'made_q2', 'made_q3',
    'driver_rolling_avg_finish', 'constructor_rolling_avg_finish',
    'driver_career_avg', 'driver_circuit_avg', 'track_delta',
    'constructor_career_avg', 'driver_vs_team_delta', 'driver_form_delta',
    'has_circuit_history', 'air_temp', 'track_temp', 'rainfall', 'humidity'
]

df[feature_check].isnull().sum()

grid_position                       2
quali_gap_to_pole                   0
made_q2                             0
made_q3                             0
driver_rolling_avg_finish          31
constructor_rolling_avg_finish     12
driver_career_avg                  31
driver_circuit_avg                730
track_delta                       730
constructor_career_avg             12
driver_vs_team_delta               33
driver_form_delta                  31
has_circuit_history                 0
air_temp                            0
track_temp                          0
rainfall                            0
humidity                            0
dtype: int64

In [37]:
overall_mean_finish = df['finish_position'].mean()

df['grid_position'] = df['grid_position'].fillna(20)
df['driver_rolling_avg_finish'] = df['driver_rolling_avg_finish'].fillna(overall_mean_finish)
df['driver_career_avg'] = df['driver_career_avg'].fillna(overall_mean_finish)
df['constructor_rolling_avg_finish'] = df['constructor_rolling_avg_finish'].fillna(overall_mean_finish)
df['constructor_career_avg'] = df['constructor_career_avg'].fillna(overall_mean_finish)
df['driver_circuit_avg'] = df['driver_circuit_avg'].fillna(df['driver_career_avg'])
df['track_delta'] = df['track_delta'].fillna(0)
df['driver_form_delta'] = df['driver_form_delta'].fillna(0)
df['driver_vs_team_delta'] = df['driver_vs_team_delta'].fillna(0)

print(df[feature_check].isnull().sum().sum())

0


In [38]:
feature_columns = [
    'grid_position', 'quali_gap_to_pole', 'made_q2', 'made_q3',
    'driver_rolling_avg_finish', 'constructor_rolling_avg_finish',
    'driver_career_avg', 'driver_circuit_avg', 'track_delta',
    'constructor_career_avg', 'driver_vs_team_delta', 'driver_form_delta',
    'has_circuit_history',
    'air_temp', 'track_temp', 'rainfall', 'humidity',
    'constructor_id', 'circuit_id'
]

model_df = df[feature_columns + ['points_finish', 'season', 'round', 'date']].copy()

print(model_df.isnull().sum().sum())
print(model_df.shape)

0
(1838, 23)


In [39]:
print(model_df.shape)

(1838, 23)


In [40]:
numeric_features = [
    'grid_position', 'quali_gap_to_pole', 'made_q2', 'made_q3',
    'driver_rolling_avg_finish', 'constructor_rolling_avg_finish',
    'driver_career_avg', 'driver_circuit_avg', 'track_delta',
    'constructor_career_avg', 'driver_vs_team_delta', 'driver_form_delta',
    'has_circuit_history', 'air_temp', 'track_temp', 'rainfall', 'humidity'
]

correlations = model_df[numeric_features + ['points_finish']].corr()['points_finish'].drop('points_finish').sort_values(ascending=False)
print(correlations)

made_q3                           0.566021
made_q2                           0.403039
has_circuit_history               0.098731
track_delta                       0.044798
track_temp                        0.000983
air_temp                          0.000243
humidity                         -0.000269
rainfall                         -0.000610
driver_vs_team_delta             -0.078785
driver_form_delta                -0.138742
quali_gap_to_pole                -0.281091
driver_circuit_avg               -0.441858
constructor_rolling_avg_finish   -0.490240
constructor_career_avg           -0.505975
driver_rolling_avg_finish        -0.514658
driver_career_avg                -0.531572
grid_position                    -0.557053
Name: points_finish, dtype: float64


In [41]:
train = model_df[model_df['season'].isin([2022, 2023, 2024])].copy()
test = model_df[model_df['season'] == 2025].copy()

print(f"Train: {train.shape}")
print(f"Test: {test.shape}")

Train: (1359, 23)
Test: (479, 23)


In [42]:
combined = pd.concat([train, test], keys=['train', 'test'])
combined_encoded = pd.get_dummies(combined, columns=['constructor_id', 'circuit_id'], drop_first=True)

train_encoded = combined_encoded.loc['train']
test_encoded = combined_encoded.loc['test']

exclude_cols = ['points_finish', 'season', 'round', 'date']
final_feature_cols = [col for col in train_encoded.columns if col not in exclude_cols]

X_train = train_encoded[final_feature_cols]
y_train = train_encoded['points_finish']
X_test = test_encoded[final_feature_cols]
y_test = test_encoded['points_finish']

print(X_train.shape, X_test.shape)

(1359, 53) (479, 53)


In [44]:
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [45]:
baseline_model = xgb.XGBClassifier(random_state=42)
baseline_model.fit(X_train[['grid_position']], y_train)
baseline_predictions = baseline_model.predict(X_test[['grid_position']])
baseline_accuracy = accuracy_score(y_test, baseline_predictions)

full_model = xgb.XGBClassifier(random_state=42)
full_model.fit(X_train, y_train)
full_predictions = full_model.predict(X_test)
full_accuracy = accuracy_score(y_test, full_predictions)

made_q3_model = xgb.XGBClassifier(random_state=42)
made_q3_model.fit(X_train[['made_q3']], y_train)
made_q3_predictions = made_q3_model.predict(X_test[['made_q3']])
made_q3_accuracy = accuracy_score(y_test, made_q3_predictions)

print(f"Baseline (grid_position only): {baseline_accuracy:.3f}")
print(f"made_q3 only: {made_q3_accuracy:.3f}")
print(f"Full model ({X_train.shape[1]} features): {full_accuracy:.3f}")

Baseline (grid_position only): 0.766
made_q3 only: 0.779
Full model (53 features): 0.739


In [46]:
best_features = ['made_q3', 'has_circuit_history', 'track_delta', 'driver_vs_team_delta']

best_model = xgb.XGBClassifier(random_state=42)
best_model.fit(X_train[best_features], y_train)
best_predictions = best_model.predict(X_test[best_features])
best_accuracy = accuracy_score(y_test, best_predictions)

print(f"made_q3 + independent features: {best_accuracy:.3f}")

made_q3 + independent features: 0.712


In [47]:
final_model = xgb.XGBClassifier(random_state=42)
final_model.fit(X_train[['made_q3']], y_train)
final_predictions = final_model.predict(X_test[['made_q3']])
final_accuracy = accuracy_score(y_test, final_predictions)

print(f"Final model accuracy: {final_accuracy:.3f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, final_predictions))
print("\nClassification Report:")
print(classification_report(y_test, final_predictions))

Final model accuracy: 0.779

Confusion Matrix:
[[188  51]
 [ 55 185]]

Classification Report:
              precision    recall  f1-score   support

       False       0.77      0.79      0.78       239
        True       0.78      0.77      0.78       240

    accuracy                           0.78       479
   macro avg       0.78      0.78      0.78       479
weighted avg       0.78      0.78      0.78       479

